In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention,196,0,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,intervention,196,0,0.0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,2,intervention,196,0,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,intervention,196,0,0.0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,3,intervention,196,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,ylls,cause,other_causes,other_causes,95_plus,severe,3,intervention,73,0,0.0
539996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,intervention,73,0,0.0
539997,ylls,cause,other_causes,other_causes,95_plus,severe,4,intervention,73,0,0.0
539998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,intervention,73,0,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  475986.426296
                                  2                  476504.466856
                                  3                  391476.381314
                                  4                  305857.410474
                                  5                  286097.042112
intervention  maternal_disorders  1                  462211.048401
                                  2                  459393.078439
                                  3                  378640.721911
                                  4                  296983.472528
                                  5                  278807.823624
zero          maternal_disorders  1                  475986.426296
                                  2                  476504.466856
                                  3                  391476.381314
                                  4                  305857.410474
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,1,intervention,196,0,1.339908
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,intervention,196,0,0.000000
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,intervention,196,0,0.000000
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,intervention,196,0,0.000000
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,intervention,196,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
1889995,ylds,cause,pregnancy,parturition,95_plus,severe,5,intervention,73,0,0.000000
1889996,ylds,cause,pregnancy,postpartum,95_plus,severe,5,intervention,73,0,0.000000
1889997,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,intervention,73,0,0.000000
1889998,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,intervention,73,0,0.000000


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  36762.346735
                                  2                  42868.451633
                                  3                  30037.144587
                                  4                  20986.067381
                                  5                  15679.343345
              maternal_disorders  1                     61.761727
                                  2                     59.471407
                                  3                     50.276951
                                  4                     40.304654
                                  5                     40.215044
intervention  anemia              1                  30614.878545
                                  2                  35757.865321
                                  3                  25034.449321
                                  4                  17805.272319
                          

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   36762.346735
                                  2                   42868.451633
                                  3                   30037.144587
                                  4                   20986.067381
                                  5                   15679.343345
              maternal_disorders  1                  476048.188024
                                  2                  476563.938263
                                  3                  391526.658264
                                  4                  305897.715128
                                  5                  286137.257156
intervention  anemia              1                   30614.878545
                                  2                   35757.865321
                                  3                   25034.449321
                                  4                   17805.272319
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,baseline,163,0,0.000000
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,baseline,163,0,0.000000
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,baseline,163,0,0.000000
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,baseline,163,0,0.000000
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,baseline,163,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
47995,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,baseline,19,0,14178.603424
47996,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,baseline,19,0,12815.808549
47997,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,baseline,19,0,11081.025958
47998,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,baseline,19,0,9882.670220


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.824815e+07
                      2                  1.844265e+07
                      3                  1.535596e+07
                      4                  1.232089e+07
                      5                  1.075245e+07
intervention  lbwsg   1                  1.817424e+07
                      2                  1.838109e+07
                      3                  1.531996e+07
                      4                  1.229894e+07
                      5                  1.073792e+07
zero          lbwsg   1                  1.824815e+07
                      2                  1.844265e+07
                      3                  1.535596e+07
                      4                  1.232089e+07
                      5                  1.075245e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,736.111714,zero
1,Female,0.0,0.019178,2,650.757670,zero
2,Female,0.0,0.019178,3,521.550200,zero
3,Female,0.0,0.019178,4,424.729902,zero
4,Female,0.0,0.019178,5,291.089988,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,80.041483,intervention
746,Male,95.0,125.000000,2,64.926221,intervention
747,Male,95.0,125.000000,3,65.384278,intervention
748,Male,95.0,125.000000,4,68.233035,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  307994.594843
                      2                  279566.887530
                      3                  268616.723697
                      4                  268525.738622
                      5                  219289.343020
intervention  anemia  1                  253917.553357
                      2                  226444.663359
                      3                  216461.545135
                      4                  213983.440031
                      5                  170883.028697
zero          anemia  1                  307994.594843
                      2                  279566.887530
                      3                  268616.723697
                      4                  268525.738622
                      5                  219289.343020
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

0.0

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  292797.033684
                      2                  251374.620768
                      3                  186307.822610
                      4                  157579.612415
                      5                   92910.679976
intervention  anemia  1                  252389.838553
                      2                  213300.364303
                      3                  155050.721581
                      4                  129476.082073
                      5                   72716.858436
zero          anemia  1                  292797.033684
                      2                  251374.620768
                      3                  186307.822610
                      4                  157579.612415
                      5                   92910.679976
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

0.0

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  955058.158686
                      2                  804660.940071
                      3                  691583.659226
                      4                  635249.935582
                      5                  449549.169032
intervention  anemia  1                  794232.576816
                      2                  658425.276738
                      3                  559075.369664
                      4                  507202.631943
                      5                  348994.343587
zero          anemia  1                  955058.158686
                      2                  804660.940071
                      3                  691583.659226
                      4                  635249.935582
                      5                  449549.169032
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  429022.091695
                      2                  438701.791378
                      3                  397145.839291
                      4                  352029.537895
                      5                  309194.085275
baseline      ntd     1                  428092.440821
                      2                  437751.165498
                      3                  396285.261284
                      4                  351266.722707
                      5                  308524.090519
intervention  ntd     1                  150610.998026
                      2                  163980.663829
                      3                  182000.903703
                      4                  182175.855550
                      5                  168048.477634
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  9.918205e+05
                                  2                  8.475294e+05
                                  3                  7.216208e+05
                                  4                  6.562360e+05
                                  5                  4.652285e+05
              lbwsg               1                  1.824815e+07
                                  2                  1.844265e+07
                                  3                  1.535596e+07
                                  4                  1.232089e+07
                                  5                  1.075245e+07
              maternal_disorders  1                  4.760482e+05
                                  2                  4.765639e+05
                                  3                  3.915267e+05
                                  4                  3.058977e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)